In [16]:
import os, random, math, hashlib
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


import random




In [17]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cuda


In [18]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_ROOT = Path("/kaggle/input")

TARGET_SIZE = 299
MAX_RESIZE = 320

BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-4
NUM_WORKERS = 2  


P_REAL = 0.50
P_GAN  = 0.25
P_DIFF = 0.25


In [19]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

def rglob_images(p: Path) -> List[Path]:
    if not p.exists():
        return []
    return [x for x in p.rglob("*") if x.suffix.lower() in IMG_EXTS]

def find_input_dir(containing: str) -> Path | None:
    """
    Search under /kaggle/input for a folder whose path contains `containing`.
    Returns the first match.
    """
    containing = containing.lower()
    for root, dirs, files in os.walk(DATASET_ROOT):
        rp = Path(root)
        if containing in str(rp).lower():
            return rp
    return None

def must_find_dir(containing: str) -> Path:
    p = find_input_dir(containing)
    if p is None:
        raise FileNotFoundError(f"Could not find any folder under {DATASET_ROOT} containing: {containing}")
    return p


In [20]:
def find_subdir_by_suffix(suffix: str) -> Path:
    suffix = suffix.replace("\\", "/")
    for p in DATASET_ROOT.rglob("*"):
        if p.is_dir() and str(p).replace("\\", "/").endswith(suffix):
            return p
    raise FileNotFoundError(f"Could not find directory ending with: {suffix}")

def build_index():
    stylegan_real_dir = find_subdir_by_suffix("train/real")
    stylegan_fake_dir = find_subdir_by_suffix("train/fake")

    stylegan_real = rglob_images(stylegan_real_dir)
    stylegan_fake = rglob_images(stylegan_fake_dir)

    rvf_real_dir = find_subdir_by_suffix("Real")
    rvf_fake_dir = find_subdir_by_suffix("Fake")

    rvf_real = rglob_images(rvf_real_dir)
    rvf_fake = rglob_images(rvf_fake_dir)

    synth_root = must_find_dir("syntheticeye-diffusion-faces")
    synth_imgs = rglob_images(synth_root)

    sd_root = find_input_dir("stable-diffusion-face-dataset")
    sd_imgs = rglob_images(sd_root) if sd_root else []

    index = {
        "real": stylegan_real + rvf_real,
        "gan": stylegan_fake,
        "diff": rvf_fake + synth_imgs,
        "cross_diff": sd_imgs
    }

    print("\n✅ DATASET INDEX SUMMARY")
    print(f"  REAL        : {len(index['real'])}")
    print(f"  GAN         : {len(index['gan'])}")
    print(f"  DIFFUSION   : {len(index['diff'])}")
    print(f"  CROSS-GEN   : {len(index['cross_diff'])}")

    return index


index = build_index()



✅ DATASET INDEX SUMMARY
  REAL        : 131000
  GAN         : 50000
  DIFFUSION   : 195550
  CROSS-GEN   : 9001


In [27]:
def build_binary_splits(index):
    splits = {"train": [], "val": [], "test_id": []}

    for group in ["real", "gan", "diff"]:
        label = 0 if group == "real" else 1
        paths = index[group][:]

        random.shuffle(paths)
        n = len(paths)
        t = int(0.7 * n)
        v = int(0.1 * n)

        splits["train"]   += [(p, label) for p in paths[:t]]
        splits["val"]     += [(p, label) for p in paths[t:t+v]]
        splits["test_id"] += [(p, label) for p in paths[t+v:]]

    crossgen = [(p, 1) for p in index["cross_diff"]]

    print("✅ BINARY SPLITS")
    print("Train:", len(splits["train"]))
    print("Val  :", len(splits["val"]))
    print("Test :", len(splits["test_id"]))
    print("Cross:", len(crossgen))

    return splits, crossgen

binary_splits, crossgen = build_binary_splits(index)


✅ BINARY SPLITS
Train: 263585
Val  : 37655
Test : 75310
Cross: 9001


In [22]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

print("✅ FFT model loaded successfully")
print("Metadata:", {k: v for k, v in checkpoint.items() if k != "model_state_dict"})


✅ FFT model loaded successfully
Metadata: {'architecture': 'resnet18', 'input': 'FFT magnitude, 299x299', 'classes': {0: 'REAL', 1: 'FAKE'}}


In [23]:
def fft_transform(img_tensor):

    gray = img_tensor.mean(dim=0, keepdim=True)
    fft = torch.fft.fft2(gray)
    fft = torch.fft.fftshift(fft)
    mag = torch.log1p(torch.abs(fft))
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return mag.repeat(3, 1, 1)


In [24]:
def frequency_mask(fft_img, p=0.5):
    if random.random() > p:
        return fft_img

    _, H, W = fft_img.shape
    cx, cy = H // 2, W // 2
    r = random.randint(int(0.05 * H), int(0.2 * H))

    Y, X = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    mask = ((X - cx)**2 + (Y - cy)**2 > r**2).float()
    return fft_img * mask


In [25]:
class FFTDebiasedDataset(Dataset):
    def __init__(self, samples, size=299):
        self.samples = samples
        self.size = size

    def __len__(self):
        return len(self.samples)

    def jpeg_compress(self, img):
        buf = io.BytesIO()
        q = random.randint(30, 90)
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")
        img = img.resize((320, 320), Image.BICUBIC)





        if label == 0:  
            if random.random() < 0.5:
                img = self.jpeg_compress(img)
            if random.random() < 0.3:
                img = img.filter(ImageFilter.UnsharpMask(radius=2, percent=150))

        else:  
            if random.random() < 0.3:
                img = img.filter(ImageFilter.GaussianBlur(radius=1.5))


        left = (320 - self.size) // 2
        img = img.crop((left, left, left+self.size, left+self.size))

        img = torch.from_numpy(np.array(img)).permute(2,0,1).float() / 255.0

        fft_img = fft_transform(img)
        fft_img = frequency_mask(fft_img)

        return fft_img, torch.tensor(label, dtype=torch.long)


In [28]:
train_ds = FFTDebiasedDataset(binary_splits["train"])
val_ds   = FFTDebiasedDataset(binary_splits["val"])
test_ds  = FFTDebiasedDataset(binary_splits["test_id"])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)


In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)  


In [30]:
def train_one_epoch():
    model.train()
    total_loss = 0
    for x,y in tqdm(train_loader):
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


In [31]:
@torch.no_grad()
def evaluate(loader, name):
    model.eval()
    ys, ps = [], []
    for x,y in tqdm(loader):
        x = x.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(1)
        ys.extend(y.numpy())
        ps.extend(preds.cpu().numpy())

    acc = accuracy_score(ys, ps)
    prec = precision_score(ys, ps)
    rec = recall_score(ys, ps)
    f1 = f1_score(ys, ps)

    print(f"{name}: acc={acc:.4f}, prec={prec:.4f}, rec={rec:.4f}, f1={f1:.4f}")


In [32]:
EPOCHS = 4

for e in range(1, EPOCHS+1):
    loss = train_one_epoch()
    print(f"\nEpoch {e} | Train loss: {loss:.4f}")
    evaluate(val_loader, "VAL")


100%|██████████| 4119/4119 [53:39<00:00,  1.28it/s] 



Epoch 1 | Train loss: 0.2532


100%|██████████| 589/589 [06:55<00:00,  1.42it/s]


VAL: acc=0.8862, prec=0.8653, rec=0.9777, f1=0.9181


100%|██████████| 4119/4119 [37:24<00:00,  1.84it/s]


...

VAL: acc=0.8861, prec=0.8609, rec=0.9845, f1=0.9185


100%|██████████| 4119/4119 [37:37<00:00,  1.82it/s]



Epoch 4 | Train loss: 0.2213


100%|██████████| 589/589 [04:52<00:00,  2.02it/s]

VAL: acc=0.8893, prec=0.8676, rec=0.9798, f1=0.9203


In [33]:
print("\nFINAL TEST")
evaluate(test_loader, "TEST_ID")



FINAL TEST


100%|██████████| 1177/1177 [12:17<00:00,  1.60it/s]


TEST_ID: acc=0.8902, prec=0.8688, rec=0.9797, f1=0.9209


In [37]:
cross_ds = FFTDebiasedDataset(crossgen)
cross_loader = DataLoader(cross_ds, batch_size=BATCH_SIZE, shuffle=False)
evaluate(cross_loader, "TEST_CROSSGEN")

100%|██████████| 141/141 [06:39<00:00,  2.84s/it]

TEST_CROSSGEN: acc=0.9621, prec=1.0000, rec=0.9621, f1=0.9807


In [34]:
SAVE_PATH = "/kaggle/working/fft_binary_resnet18_debiased.pth"
torch.save(model.state_dict(), SAVE_PATH)
print("✅ Saved debiased model:", SAVE_PATH)


✅ Saved debiased model: /kaggle/working/fft_binary_resnet18_debiased.pth


/kaggle/input/frequency-model-checkpoint/fft_binary_resnet18.pth
